# Module 4 — Question Answering with Pretrained Transformer
**Pipeline:** Fine-tuning → **DistilBERT on SQuAD 2.0**  
**Dataset:** SQuAD 2.0 (answerable + unanswerable)  
**Model:** DistilBERT-base-uncased (fine-tuned from scratch on SQuAD 2.0)  
**Evaluation:** Exact Match (EM) + F1  
**Comparison:** BiLSTM + GloVe (Module 3) vs DistilBERT (this notebook)

## 1. Install & Import

In [13]:
!pip install transformers datasets -q

import os
import re
import json
import string
import collections
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from datasets import load_dataset
import warnings
warnings.filterwarnings('ignore')

assert torch.cuda.is_available(), "Enable GPU in Kaggle: Settings → Accelerator → GPU"
DEVICE = torch.device('cuda')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 2. Configuration

In [36]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME   = 'distilbert-base-uncased'   # base model, NOT already SQuAD fine-tuned

# ── Tokenization ───────────────────────────────────────────────────────────
MAX_LENGTH   = 384    # max tokens per (question + context) chunk
STRIDE       = 128    # overlap between sliding window chunks for long contexts

# ── Training ───────────────────────────────────────────────────────────────
BATCH_SIZE   = 16     # fits comfortably on a Kaggle T4 GPU
EPOCHS       = 1     # 3 epochs is standard for transformer fine-tuning on SQuAD
LR           = 2e-5   # learning rate — standard for DistilBERT fine-tuning
WEIGHT_DECAY = 0.01   # L2 regularisation on non-bias parameters
WARMUP_RATIO = 0.10   # fraction of total steps used for linear LR warmup
GRAD_CLIP    = 1.0    # max gradient norm to prevent exploding gradients
LOG_STEPS    = 500    # print train loss every N steps

# ── Evaluation ─────────────────────────────────────────────────────────────
EVAL_SAMPLES = 1000   # number of val samples used for EM/F1 during training

# ── Paths ──────────────────────────────────────────────────────────────────
SAVE_DIR     = '/kaggle/working/distilbert_squad2'
os.makedirs(SAVE_DIR, exist_ok=True)

print('Config loaded.')

Config loaded.


## 3. Load SQuAD 2.0

In [37]:
print('Loading SQuAD 2.0...')
squad = load_dataset('rajpurkar/squad_v2')

print(f"Train examples : {len(squad['train'])}")
print(f"Val examples   : {len(squad['validation'])}")

# Quick look at one example
ex = squad['train'][0]
print(f"\nSample question : {ex['question']}")
print(f"Sample answer   : {ex['answers']}")

Loading SQuAD 2.0...
Train examples : 130319
Val examples   : 11873

Sample question : When did Beyonce start becoming popular?
Sample answer   : {'text': ['in the late 1990s'], 'answer_start': [269]}


## 4. Tokenization & Dataset

Transformer QA tokenization is more complex than BiLSTM tokenization because:
- The model takes `[CLS] question [SEP] context [SEP]` as a single sequence
- Contexts longer than MAX_LENGTH are split into overlapping chunks (stride)
- We need to map character-level answer positions → token-level positions

In [38]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess_examples(examples):
    """
    Tokenize a batch of SQuAD examples with sliding window for long contexts.
    Returns token IDs, attention masks, and token-level start/end positions.
    """
    questions = [q.strip() for q in examples['question']]
    contexts  = examples['context']

    # Tokenize — produces multiple chunks per example if context is long
    tokenized = tokenizer(
        questions,
        contexts,
        max_length            = MAX_LENGTH,
        stride                = STRIDE,
        truncation            = 'only_second',      # only truncate the context
        return_overflowing_tokens = True,           # split long contexts into chunks
        return_offsets_mapping    = True,           # char→token mapping for span finding
        padding               = 'max_length',
    )

    # Map each chunk back to its original example index
    sample_map     = tokenized.pop('overflow_to_sample_mapping')
    offset_mapping = tokenized.pop('offset_mapping')

    start_positions = []
    end_positions   = []

    for i, offsets in enumerate(offset_mapping):
        orig_idx  = sample_map[i]           # which original example this chunk belongs to
        answers   = examples['answers'][orig_idx]
        input_ids = tokenized['input_ids'][i]

        # Find the token index of [CLS] — used as the "unanswerable" position
        cls_index = input_ids.index(tokenizer.cls_token_id)

        # Determine which tokens belong to the context (not question/special tokens)
        sequence_ids = tokenized.sequence_ids(i)
        # sequence_ids: None=special, 0=question tokens, 1=context tokens

        # ── Unanswerable question ──────────────────────────────────────────
        if len(answers['text']) == 0:
            # Point both positions at [CLS] — model learns to predict CLS
            # when the question cannot be answered from the context
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # ── Answerable question ────────────────────────────────────────────
        answer_start_char = answers['answer_start'][0]
        answer_end_char   = answer_start_char + len(answers['text'][0])

        # Find the context token span within this chunk
        token_start = 0
        token_end   = 0

        # Find first context token index
        ctx_start_idx = next(
            (idx for idx, s in enumerate(sequence_ids) if s == 1), None
        )
        # Find last context token index
        ctx_end_idx = next(
            (idx for idx in range(len(sequence_ids)-1, -1, -1)
             if sequence_ids[idx] == 1), None
        )

        # If the answer is outside this chunk's context window → use CLS
        if (ctx_start_idx is None or ctx_end_idx is None or
            offsets[ctx_start_idx][0] > answer_start_char or
            offsets[ctx_end_idx][1]   < answer_end_char):
            start_positions.append(cls_index)
            end_positions.append(cls_index)
            continue

        # Walk tokens to find the exact start and end token indices
        token_start = ctx_start_idx
        while token_start <= ctx_end_idx and offsets[token_start][0] <= answer_start_char:
            token_start += 1
        start_positions.append(token_start - 1)

        token_end = ctx_end_idx
        while token_end >= ctx_start_idx and offsets[token_end][1] >= answer_end_char:
            token_end -= 1
        end_positions.append(token_end + 1)

    tokenized['start_positions'] = start_positions
    tokenized['end_positions']   = end_positions
    return tokenized


print('Tokenizing train split (full dataset, this takes ~3–5 min)...')
train_tokenized = squad['train'].map(
    preprocess_examples,
    batched      = True,
    remove_columns = squad['train'].column_names,
)

print('Tokenizing validation split...')
val_tokenized = squad['validation'].map(
    preprocess_examples,
    batched      = True,
    remove_columns = squad['validation'].column_names,
)

print(f'Train chunks : {len(train_tokenized)}')
print(f'Val chunks   : {len(val_tokenized)}')
print('(chunks > examples because long contexts are split into overlapping windows)')

Tokenizing train split (full dataset, this takes ~3–5 min)...
Tokenizing validation split...
Train chunks : 131754
Val chunks   : 12134
(chunks > examples because long contexts are split into overlapping windows)


## 5. PyTorch Dataset & DataLoader

In [39]:
class SQuADDataset(Dataset):
    def __init__(self, hf_dataset):
        self.data = hf_dataset

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        return {
            'input_ids'      : torch.tensor(item['input_ids'],       dtype=torch.long),
            'attention_mask' : torch.tensor(item['attention_mask'],  dtype=torch.long),
            'start_positions': torch.tensor(item['start_positions'], dtype=torch.long),
            'end_positions'  : torch.tensor(item['end_positions'],   dtype=torch.long),
        }


train_ds = SQuADDataset(train_tokenized)
val_ds   = SQuADDataset(val_tokenized)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train batches : {len(train_loader)}')
print(f'Val batches   : {len(val_loader)}')

Train batches : 8235
Val batches   : 759


## 6. Model, Optimizer & Scheduler

In [40]:
# Load DistilBERT with a QA head (two linear layers: start logits + end logits)
# This is the BASE model — not pre-fine-tuned on SQuAD — so all QA learning
# happens during our fine-tuning below.
model = AutoModelForQuestionAnswering.from_pretrained('/kaggle/working/distilbert_squad2_checkpoint')
if torch.cuda.device_count() > 1:
    print(f'Using {torch.cuda.device_count()} GPUs')
    model = torch.nn.DataParallel(model)

model = model.to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

# ── Optimizer ─────────────────────────────────────────────────────────────
# AdamW = Adam with weight decay fix (standard for transformer fine-tuning)
# We don't apply weight decay to bias terms or LayerNorm parameters
no_decay = ['bias', 'LayerNorm.weight']
optimizer_params = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
     'weight_decay': 0.0},
]
optimizer = AdamW(optimizer_params, lr=LR)

# ── Scheduler ─────────────────────────────────────────────────────────────
# Linear warmup for first 10% of steps, then linear decay to 0
# Warmup prevents large updates at the start when the QA head is random
total_steps  = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler    = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps   = warmup_steps,
    num_training_steps = total_steps,
)

print(f'Total training steps : {total_steps:,}')
print(f'Warmup steps         : {warmup_steps:,}')

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Using 2 GPUs
Total parameters     : 66,364,418
Trainable parameters : 66,364,418
Total training steps : 8,235
Warmup steps         : 823


## 7. Evaluation Helpers (SQuAD official metric)

In [41]:
def normalize_answer(s):
    """Lowercase, remove articles/punctuation, fix whitespace."""
    def remove_articles(t):  return re.sub(r'\b(a|an|the)\b', ' ', t)
    def white_space_fix(t):  return ' '.join(t.split())
    def remove_punc(t):      return ''.join(c for c in t if c not in set(string.punctuation))
    def lower(t):            return t.lower()
    return white_space_fix(remove_articles(remove_punc(lower(s))))

def get_tokens(s):
    return normalize_answer(s).split()

def compute_exact(a_gold, a_pred):
    return int(normalize_answer(a_gold) == normalize_answer(a_pred))

def compute_f1(a_gold, a_pred):
    gold_toks = get_tokens(a_gold)
    pred_toks = get_tokens(a_pred)
    common    = collections.Counter(gold_toks) & collections.Counter(pred_toks)
    num_same  = sum(common.values())
    if not gold_toks or not pred_toks:
        return int(gold_toks == pred_toks)
    if num_same == 0:
        return 0.0
    precision = num_same / len(pred_toks)
    recall    = num_same / len(gold_toks)
    return (2 * precision * recall) / (precision + recall)

print('Evaluation helpers ready.')

Evaluation helpers ready.


## 8. Inference Function

In [42]:
def predict_answer(question, context, model, tokenizer, max_answer_len=50):
    inputs = tokenizer(
        question, context,
        max_length=MAX_LENGTH,
        truncation='only_second',
        return_offsets_mapping=True,
        return_tensors='pt',
        padding='max_length',
    )
    offset_mapping = inputs.pop('offset_mapping')[0]
    sequence_ids   = inputs.sequence_ids(0)
    inputs         = {k: v.to(DEVICE) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    start_logits = outputs.start_logits[0].cpu().numpy()
    end_logits   = outputs.end_logits[0].cpu().numpy()

    mask  = np.array([1 if s == 1 else 0 for s in sequence_ids], dtype=bool)
    s_log = np.where(mask, start_logits, -1e9)
    e_log = np.where(mask, end_logits,   -1e9)

    scores    = s_log[:, None] + e_log[None, :]
    triu      = np.triu(np.ones((MAX_LENGTH, MAX_LENGTH), dtype=bool))
    length_ok = (np.arange(MAX_LENGTH)[None, :] -
                 np.arange(MAX_LENGTH)[:, None]) < max_answer_len
    scores    = np.where(triu & length_ok, scores, -1e9)

    cls_score  = start_logits[0] + end_logits[0]
    best_idx   = np.unravel_index(scores.argmax(), scores.shape)
    best_score = scores[best_idx]

    if best_score <= cls_score:
        return ''

    answer_ids = inputs['input_ids'][0][best_idx[0]: best_idx[1] + 1]
    return tokenizer.decode(answer_ids.cpu(), skip_special_tokens=True)

print('predict_answer() ready.')

predict_answer() ready.


## 9. Training Loop

In [43]:
# Keep a reference to the raw validation examples for EM/F1 evaluation
val_raw = list(squad['validation'])
# Filter to answerable samples for EM/F1 (same methodology as Module 3)
val_answerable = [x for x in val_raw if len(x['answers']['text']) > 0]
print(f'Answerable val examples : {len(val_answerable)}')


def train_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total_loss = 0.0
    step       = 0
    for batch in loader:
        input_ids       = batch['input_ids'].to(DEVICE)
        attention_mask  = batch['attention_mask'].to(DEVICE)
        start_positions = batch['start_positions'].to(DEVICE)
        end_positions   = batch['end_positions'].to(DEVICE)
        optimizer.zero_grad()
        outputs = model(
            input_ids       = input_ids,
            attention_mask  = attention_mask,
            start_positions = start_positions,
            end_positions   = end_positions,
        )
        loss = outputs.loss.mean()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        step       += 1
        if step % LOG_STEPS == 0:
            avg = total_loss / step
            lr  = scheduler.get_last_lr()[0]
            print(f'  Epoch {epoch} | Step {step}/{len(loader)} | Loss: {avg:.4f} | LR: {lr:.2e}')
    return total_loss / len(loader)


@torch.no_grad()
def eval_loss(model, loader):
    model.eval()
    total_loss = 0.0
    for batch in loader:
        outputs = model(
            input_ids       = batch['input_ids'].to(DEVICE),
            attention_mask  = batch['attention_mask'].to(DEVICE),
            start_positions = batch['start_positions'].to(DEVICE),
            end_positions   = batch['end_positions'].to(DEVICE),
        )
        total_loss += outputs.loss.mean().item()  # ✅ inside the loop
    return total_loss / len(loader)


def eval_em_f1(model, examples, n=EVAL_SAMPLES):
    """Run predict_answer on the first n answerable examples and compute EM/F1."""
    model.eval()
    em_scores, f1_scores = [], []
    for item in examples[:n]:
        gold = item['answers']['text'][0]
        pred = predict_answer(item['question'], item['context'], model, tokenizer)
        em_scores.append(compute_exact(gold, pred))
        f1_scores.append(compute_f1(gold, pred))
    return np.mean(em_scores), np.mean(f1_scores)


# ── Main training loop ─────────────────────────────────────────────────────
best_val_loss = float('inf')

for epoch in range(1, EPOCHS + 1):
    print(f'\n{"="*65}')
    print(f'EPOCH {epoch}/{EPOCHS}')
    print('='*65)

    tr_loss  = train_epoch(model, train_loader, optimizer, scheduler, epoch)
    val_loss = eval_loss(model, val_loader)

    print(f'\nComputing EM/F1 on {EVAL_SAMPLES} val samples...')
    val_em, val_f1 = eval_em_f1(model, val_answerable)

    print(f'Epoch {epoch}/{EPOCHS} | Train Loss: {tr_loss:.4f} | Val Loss: {val_loss:.4f} | Val EM: {val_em:.4f} | Val F1: {val_f1:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.module.save_pretrained(SAVE_DIR)
        tokenizer.save_pretrained(SAVE_DIR)
        print(f'  ✓ Saved best model (val loss {val_loss:.4f})')

print('\nTraining complete.')

Answerable val examples : 5928

EPOCH 1/1
  Epoch 1 | Step 500/8235 | Loss: 0.7806 | LR: 1.22e-05
  Epoch 1 | Step 1000/8235 | Loss: 0.8117 | LR: 1.95e-05
  Epoch 1 | Step 1500/8235 | Loss: 0.8236 | LR: 1.82e-05
  Epoch 1 | Step 2000/8235 | Loss: 0.8329 | LR: 1.68e-05
  Epoch 1 | Step 2500/8235 | Loss: 0.8353 | LR: 1.55e-05
  Epoch 1 | Step 3000/8235 | Loss: 0.8378 | LR: 1.41e-05
  Epoch 1 | Step 3500/8235 | Loss: 0.8384 | LR: 1.28e-05
  Epoch 1 | Step 4000/8235 | Loss: 0.8402 | LR: 1.14e-05
  Epoch 1 | Step 4500/8235 | Loss: 0.8383 | LR: 1.01e-05
  Epoch 1 | Step 5000/8235 | Loss: 0.8361 | LR: 8.73e-06
  Epoch 1 | Step 5500/8235 | Loss: 0.8326 | LR: 7.38e-06
  Epoch 1 | Step 6000/8235 | Loss: 0.8316 | LR: 6.03e-06
  Epoch 1 | Step 6500/8235 | Loss: 0.8292 | LR: 4.68e-06
  Epoch 1 | Step 7000/8235 | Loss: 0.8274 | LR: 3.33e-06
  Epoch 1 | Step 7500/8235 | Loss: 0.8241 | LR: 1.98e-06
  Epoch 1 | Step 8000/8235 | Loss: 0.8227 | LR: 6.34e-07

Computing EM/F1 on 1000 val samples...
Epoch 1

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

  ✓ Saved best model (val loss 1.4155)

Training complete.


## 10. Final Evaluation — SQuAD 2.0 Full Validation Set

In [44]:
# Load best checkpoint
print('Loading best model checkpoint...')
model     = AutoModelForQuestionAnswering.from_pretrained(SAVE_DIR).to(DEVICE)
tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)

# Evaluate on ALL answerable validation examples
print(f'Evaluating on all {len(val_answerable)} answerable val examples...')
print('(This takes ~5–10 min — one forward pass per example)')

em_scores, f1_scores = [], []
for i, item in enumerate(val_answerable):
    gold = item['answers']['text'][0]
    pred = predict_answer(item['question'], item['context'], model, tokenizer)
    em_scores.append(compute_exact(gold, pred))
    f1_scores.append(compute_f1(gold, pred))
    if (i + 1) % 500 == 0:
        print(f'  {i+1}/{len(val_answerable)} | EM so far: {np.mean(em_scores):.4f} | F1 so far: {np.mean(f1_scores):.4f}')

final_em = np.mean(em_scores)
final_f1 = np.mean(f1_scores)
print(f'\nSQuAD 2.0 Final | EM: {final_em:.4f} | F1: {final_f1:.4f}')

Loading best model checkpoint...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Evaluating on all 5928 answerable val examples...
(This takes ~5–10 min — one forward pass per example)
  500/5928 | EM so far: 0.4860 | F1 so far: 0.5938
  1000/5928 | EM so far: 0.5010 | F1 so far: 0.6146
  1500/5928 | EM so far: 0.4947 | F1 so far: 0.6150
  2000/5928 | EM so far: 0.4870 | F1 so far: 0.6185
  2500/5928 | EM so far: 0.4684 | F1 so far: 0.6104
  3000/5928 | EM so far: 0.4567 | F1 so far: 0.5995
  3500/5928 | EM so far: 0.4537 | F1 so far: 0.5950
  4000/5928 | EM so far: 0.4728 | F1 so far: 0.6090
  4500/5928 | EM so far: 0.4740 | F1 so far: 0.6124
  5000/5928 | EM so far: 0.4768 | F1 so far: 0.6142
  5500/5928 | EM so far: 0.4796 | F1 so far: 0.6183

SQuAD 2.0 Final | EM: 0.4685 | F1: 0.6072


## 11. Zero-Shot Evaluation — HotpotQA (Multi-Hop)

In [45]:
print('Loading HotpotQA (distractor setting)...')
hotpot = load_dataset('hotpot_qa', 'distractor')

def build_hotpot_context(item):
    """Concatenate all supporting paragraph sentences into one context string."""
    parts = []
    for sentences in item['context']['sentences']:
        parts.append(' '.join(sentences))
    return ' '.join(parts)

HOTPOT_SAMPLES = 300
em_hot, f1_hot = [], []

print(f'Evaluating on {HOTPOT_SAMPLES} HotpotQA val examples (zero-shot)...')
for item in list(hotpot['validation'])[:HOTPOT_SAMPLES]:
    context = build_hotpot_context(item)
    gold    = item['answer']
    pred    = predict_answer(item['question'], context, model, tokenizer)
    em_hot.append(compute_exact(gold, pred))
    f1_hot.append(compute_f1(gold, pred))

hotpot_em = np.mean(em_hot)
hotpot_f1 = np.mean(f1_hot)
print(f'HotpotQA (zero-shot) | EM: {hotpot_em:.4f} | F1: {hotpot_f1:.4f}')

Loading HotpotQA (distractor setting)...


README.md: 0.00B [00:00, ?B/s]

distractor/train-00000-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/train-00001-of-00002.parquet:   0%|          | 0.00/166M [00:00<?, ?B/s]

distractor/validation-00000-of-00001.par(…):   0%|          | 0.00/27.5M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/90447 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/7405 [00:00<?, ? examples/s]

Evaluating on 300 HotpotQA val examples (zero-shot)...
HotpotQA (zero-shot) | EM: 0.0233 | F1: 0.0636


## 12. Quick Demo — Pipeline

In [46]:
demo_pairs = [
    {
        'question': 'Who invented the telephone?',
        'context' : 'Alexander Graham Bell is credited with inventing the telephone in 1876. '
                    'He was a Scottish-born scientist and inventor who later became a US citizen.',
    },
    {
        'question': 'When did World War II end?',
        'context' : 'World War II ended in 1945. Germany surrendered on May 8, 1945 (V-E Day), '
                    'and Japan surrendered on September 2, 1945 (V-J Day).',
    },
    {
        'question': 'What is the boiling point of water on Mars?',
        'context' : 'The Eiffel Tower was built in Paris between 1887 and 1889. '
                    'It stands 330 meters tall and was designed by Gustave Eiffel.',
    },
]

print('Demo predictions:')
print('='*65)
for pair in demo_pairs:
    pred = predict_answer(pair['question'], pair['context'], model, tokenizer)
    print(f"Q : {pair['question']}")
    print(f"A : {pred if pred else '[unanswerable]'}")
    print('-'*65)

Demo predictions:
Q : Who invented the telephone?
A : alexander graham bell
-----------------------------------------------------------------
Q : When did World War II end?
A : 1945
-----------------------------------------------------------------
Q : What is the boiling point of water on Mars?
A : [unanswerable]
-----------------------------------------------------------------


## 13. Results Summary

In [47]:
print()
print('='*65)
print('         FINAL RESULTS SUMMARY')
print('='*65)
print()
print(f'{"Model":<35} {"EM":>8} {"F1":>8}')
print('-'*55)

# Module 3 results (from scratch)
print(f'{"BiLSTM + GloVe (from scratch)":<35} {"~10%":>8} {"~17%":>8}')

# This notebook results
print(f'{"DistilBERT fine-tuned (SQuAD 2.0)":<35} {final_em*100:>7.1f}% {final_f1*100:>7.1f}%')

print()
print('Zero-shot transfer (HotpotQA multi-hop):')
print(f'{"":<35} {"EM":>8} {"F1":>8}')
print('-'*55)
print(f'{"BiLSTM + GloVe (from scratch)":<35} {"low":>8} {"low":>8}')
print(f'{"DistilBERT fine-tuned":<35} {hotpot_em*100:>7.1f}% {hotpot_f1*100:>7.1f}%')

print()
print('Notes:')
print('  - BiLSTM trained on 20k SQuAD 2.0 samples, GloVe 100d embeddings')
print('  - DistilBERT fine-tuned on full SQuAD 2.0 (~87k train samples)')
print('  - Same EM/F1 evaluation functions used in both notebooks')
print('='*65)


         FINAL RESULTS SUMMARY

Model                                     EM       F1
-------------------------------------------------------
BiLSTM + GloVe (from scratch)           ~10%     ~17%
DistilBERT fine-tuned (SQuAD 2.0)      46.8%    60.7%

Zero-shot transfer (HotpotQA multi-hop):
                                          EM       F1
-------------------------------------------------------
BiLSTM + GloVe (from scratch)            low      low
DistilBERT fine-tuned                   2.3%     6.4%

Notes:
  - BiLSTM trained on 20k SQuAD 2.0 samples, GloVe 100d embeddings
  - DistilBERT fine-tuned on full SQuAD 2.0 (~87k train samples)
  - Same EM/F1 evaluation functions used in both notebooks


In [29]:
# Save epoch 1 checkpoint before restarting
if hasattr(model, 'module'):
    model.module.save_pretrained('/kaggle/working/distilbert_squad2_checkpoint')
else:
    model.save_pretrained('/kaggle/working/distilbert_squad2_checkpoint')

tokenizer.save_pretrained('/kaggle/working/distilbert_squad2_checkpoint')
print('Checkpoint saved.')

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Checkpoint saved.
